# tqdm-postfix-metrics — ex1: set live loss + examples_seen on a tqdm progress bar

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tqdm-postfix-metrics`. Running the final beacon cell reports progress against the `Logging: tqdm postfix metrics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: tqdm postfix metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tqdm-postfix-metrics`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tqdm-postfix-metrics"
DD_SUBTOPIC = "Logging: tqdm postfix metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `pbar.set_postfix(loss=..., ex=...)` — quick refresher

`tqdm` is the progress-bar library used in every ARENA training loop. Wrapping a DataLoader in `tqdm(...)` gives you the iterating bar; `pbar.set_postfix(**metrics)` then appends key=value pairs to the RIGHT side of that bar so you can see live metrics scroll while the loop runs.

**The two-step idiom:**

```python
pbar = tqdm(self.train_loader, desc='Training')
for imgs, labels in pbar:
    loss = self.training_step(imgs, labels)
    pbar.set_postfix(loss=f'{loss:.3f}', ex_seen=self.examples_seen)
```

**`set_postfix` overwrites previous postfix** — each call REPLACES the metrics dict, it doesn't accumulate. Pass every metric you want on the bar in every call.

**Format numbers in the call.** `loss=f'{loss:.3f}'` prints `0.412` not `0.41234567...`. Pass `int(ex_seen)` if you want no decimal places. tqdm doesn't auto-format floats nicely.

**Different from wandb.log.** `set_postfix` updates the LOCAL terminal bar; `wandb.log` ships metrics to the cloud dashboard. You typically call both in the same step.

### Exercise 1 — set live loss + examples_seen on a tqdm progress bar

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `tqdm(iterable, desc=...)` + `pbar.set_postfix(**metrics)` to display live per-step metrics on a progress bar while iterating a fake DataLoader.
> Keywords: tqdm, postfix, progress-bar, live-metrics
> ```

**KCs targeted:** `tqdm-wrap-iterable`, `tqdm-set-postfix-kwargs`

Implement `ex1_run_with_tqdm_postfix(losses, batch_size)`. The canonical ARENA inner-loop progress-bar idiom (without wandb):

1. Wrap `enumerate(losses)` in `tqdm(...)` with `desc='Training'`. Capture the wrapped object — you need it to call `set_postfix`.
2. For each `(step, loss)` pair from the wrapped iterator:
   - Accumulate `examples_seen += batch_size`.
   - Call `pbar.set_postfix(loss=f'{loss:.3f}', examples_seen=examples_seen)`.
3. Return the list of postfix dicts that were set (`pbar.postfix` is set, but tqdm stores it differently across versions — record it yourself into a sidecar list so the test can inspect).

**Wrapping `enumerate(losses)`:** `for step, loss in tqdm(...)` is exactly what ARENA writes. Don't fight it — wrap the enumerate, not the bare losses.

In [ ]:
from tqdm import tqdm

def ex1_run_with_tqdm_postfix(losses: list, batch_size: int) -> list:
    """Iterate losses with a tqdm bar, set per-step postfix. Return list of postfix dicts."""
    raise NotImplementedError()


def _test_ex1():
    losses = [0.91, 0.75, 0.42, 0.18, 0.06]
    batch_size = 64
    postfix_log = ex1_run_with_tqdm_postfix(losses, batch_size)

    # One postfix entry per loss.
    assert len(postfix_log) == len(losses), (
        f'expected {len(losses)} postfix entries, got {len(postfix_log)}'
    )

    # Each entry is a dict with 'loss' (formatted str) and 'examples_seen' (int).
    for i, (loss, pf) in enumerate(zip(losses, postfix_log)):
        assert isinstance(pf, dict), f'entry {i} not a dict'
        assert 'loss' in pf, f'entry {i} missing "loss" key'
        assert 'examples_seen' in pf, f'entry {i} missing "examples_seen" key'
        # loss formatted as .3f.
        expected_loss_str = f'{loss:.3f}'
        assert pf['loss'] == expected_loss_str, (
            f'entry {i}: loss must be formatted as .3f — expected {expected_loss_str!r}, got {pf["loss"]!r}'
        )
        # examples_seen monotone increasing by batch_size each step.
        expected_ex = (i + 1) * batch_size
        assert pf['examples_seen'] == expected_ex, (
            f'entry {i}: examples_seen must be {expected_ex}, got {pf["examples_seen"]}'
        )

    # Empty losses → no postfix, no error.
    assert ex1_run_with_tqdm_postfix([], 32) == []
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
from tqdm import tqdm

def ex1_run_with_tqdm_postfix(losses, batch_size):
    examples_seen = 0
    postfix_log = []
    pbar = tqdm(enumerate(losses), desc='Training', total=len(losses))
    for step, loss in pbar:
        examples_seen += batch_size
        pf = dict(loss=f'{loss:.3f}', examples_seen=examples_seen)
        pbar.set_postfix(**pf)
        postfix_log.append(pf)
    return postfix_log
```

**Why record `postfix_log` separately.** tqdm's `pbar.postfix` attribute stores the LAST postfix you set, not the history. For the drill we record a sidecar list so the test can inspect every step's metrics — in real ARENA code you don't need this because wandb.log captures the history.

**Why `total=len(losses)`.** When you wrap `enumerate(losses)`, tqdm can't infer the length (enumerate is a generator). Passing `total=` makes the bar show '5/100' not '5it'. Optional for correctness, mandatory for nice UX.

**`f'{loss:.3f}'` not `loss`.** Pass a STRING to `set_postfix` when you want exact decimal control. Passing a float lets tqdm auto-format, which produces inconsistent precision (depends on the magnitude). ARENA always pre-formats.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()